# Statistical Co-Mutation Analysis (Fisher's exact test / Haldane–Anscombe / Benjamini–Hochberg)

Implements the pairwise gene co-mutation test described in the Methods ("Statistical Co-Mutation Analysis" section): a sample-by-gene binary mutation matrix, Fisher's exact test per gene pair, Haldane–Anscombe-corrected odds ratios for zero-cell tables, and Benjamini–Hochberg FDR correction, run at the pan-cancer level, in lung cancer samples, and within primary tumor sites with sufficient sample counts.

Input: the cleaned missense table produced by `src/01_preprocess_cosmic.py` (set `SRC` below or the `COSMIC_CSV` environment variable).

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from itertools import combinations


## Gene panel and binary sample×gene matrix

In [2]:
import os
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
SRC = os.environ.get('COSMIC_CSV', os.path.join(ROOT, 'data', 'cosmic_panel_missense.csv'))
PANEL = ['GNAS','GNAQ','GNA11','GNA13','GNA14','GNA15','GNAI1','GNAI2',
         'GNB1','GNB3','GNB5','ARR3','SAG','ARRB1','ARRB2']

df = pd.read_csv(SRC)
panel_df = df[df['Gene name'].isin(PANEL)].copy()

# one Primary site per ID_sample?
site_per_sample = panel_df.groupby('ID_sample')['Primary site'].nunique()
print('samples with conflicting Primary site across rows:', (site_per_sample > 1).sum())

mat = (panel_df.assign(v=1)
       .pivot_table(index='ID_sample', columns='Gene name', values='v',
                     aggfunc='max', fill_value=0))
for g in PANEL:
    if g not in mat.columns:
        mat[g] = 0
mat = mat[PANEL]

site_map = panel_df.drop_duplicates('ID_sample').set_index('ID_sample')['Primary site']
mat['Primary site'] = mat.index.map(site_map)
print('unique panel samples:', len(mat))
mat.head()


samples with conflicting Primary site across rows: 0
unique panel samples: 6152


Gene name,GNAS,GNAQ,GNA11,GNA13,GNA14,GNA15,GNAI1,GNAI2,GNB1,GNB3,GNB5,ARR3,SAG,ARRB1,ARRB2,Primary site
ID_sample,,,,,,,,,,,,,,,,
684681,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,lung
753546,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,skin
753596,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,skin
1098123,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,thyroid
1099512,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,thyroid


## Pairwise Fisher's exact test + Haldane–Anscombe OR, per stratum

In [3]:
def run_pairwise(sub_mat, label):
    n = len(sub_mat)
    rows = []
    for g1, g2 in combinations(PANEL, 2):
        a = int(((sub_mat[g1]==1) & (sub_mat[g2]==1)).sum())   # both mutated
        b = int(((sub_mat[g1]==1) & (sub_mat[g2]==0)).sum())   # g1 only
        c = int(((sub_mat[g1]==0) & (sub_mat[g2]==1)).sum())   # g2 only
        e = int(((sub_mat[g1]==0) & (sub_mat[g2]==0)).sum())   # neither
        table = [[a,b],[c,e]]
        try:
            odds_raw, p = fisher_exact(table, alternative='two-sided')
        except Exception:
            odds_raw, p = np.nan, np.nan
        if min(a,b,c,e) == 0:
            ha = ((a+0.5)*(e+0.5)) / ((b+0.5)*(c+0.5))   # Haldane-Anscombe correction
            ha_flag = True
        else:
            ha = (a*e)/(b*c)
            ha_flag = False
        rows.append({
            'stratum': label, 'n_samples': n, 'gene1': g1, 'gene2': g2,
            'both_mut': a, 'gene1_only': b, 'gene2_only': c, 'neither': e,
            'odds_ratio_raw': odds_raw, 'odds_ratio_HA': ha, 'HA_applied': ha_flag,
            'p_value': p
        })
    res = pd.DataFrame(rows)
    if len(res):
        res['q_value_BH'] = multipletests(res['p_value'], method='fdr_bh')[1]
        res['direction'] = np.where(res['odds_ratio_HA']>1, 'co-occurrence', 'mutual exclusivity')
    return res


## Run at pan-cancer level, in lung samples, and within primary sites with ≥50 samples

In [4]:
all_results = [run_pairwise(mat, 'pan-cancer')]

lung = mat[mat['Primary site']=='lung']
all_results.append(run_pairwise(lung, 'lung'))

site_counts = mat['Primary site'].value_counts()
qualifying_sites = site_counts[site_counts>=50].index.tolist()
for site in qualifying_sites:
    if site == 'lung':
        continue
    sub = mat[mat['Primary site']==site]
    all_results.append(run_pairwise(sub, f'site:{site}'))

full = pd.concat(all_results, ignore_index=True)
os.makedirs(os.path.join(ROOT, 'results'), exist_ok=True)
full.to_csv(os.path.join(ROOT, 'results', 'CoMutation_Statistical_Results_FULL.csv'), index=False)
print('strata tested:', 1 + 1 + len(qualifying_sites) - 1, ' | total pairwise tests:', len(full))


strata tested: 27  | total pairwise tests: 2835


## Headline result: ARRB1–ARRB2 (the manuscript's candidate pair) across all strata

Not significant at any of the 27 strata tested (all q ≥ 0.82).

In [5]:
arrb = full[(full['gene1']=='ARRB1')&(full['gene2']=='ARRB2')].sort_values('q_value_BH')
arrb[['stratum','n_samples','both_mut','gene1_only','gene2_only','neither',
      'odds_ratio_HA','p_value','q_value_BH','direction']].head(8)


,stratum,n_samples,both_mut,gene1_only,gene2_only,neither,odds_ratio_HA,p_value,q_value_BH,direction
419,site:skin,717,2,27,30,658,1.624691,0.375366,0.819690,co-occurrence
104,pan-cancer,6152,4,150,145,5853,1.076414,0.789043,0.888107,co-occurrence
2624,site:central_nervous_system,56,0,4,2,50,2.244444,1.000000,1.000000,co-occurrence
2519,site:NS,56,0,2,2,52,4.200000,1.000000,1.000000,co-occurrence
2414,site:meninges,58,0,0,0,58,117.000000,1.000000,1.000000,co-occurrence
2309,site:kidney,71,0,2,5,64,2.345455,1.000000,1.000000,co-occurrence
2204,site:prostate,73,0,4,4,65,1.617284,1.000000,1.000000,co-occurrence
2099,site:ovary,75,0,4,4,67,1.666667,1.000000,1.000000,co-occurrence


Pan-cancer: 4/6,152 samples carry both an ARRB1 and an ARRB2 missense mutation (OR=1.08, p=0.789, q=0.888). Lung-restricted: 1/321 (OR=1.50, p=0.519, q=1.00) — this is the single sample (ID_sample 2640423) discussed in the manuscript as carrying ARRB1 p.T224N + ARRB2 p.T268I.

## Validity check: known biology is recovered

The strongest signal in the entire run reproduces the well-established, near-mutually-exclusive GNAQ/GNA11 hotspot pattern in uveal melanoma — good evidence the pipeline is implemented correctly.

In [6]:
full.sort_values('q_value_BH').head(5)[['stratum','gene1','gene2','both_mut','gene1_only',
    'gene2_only','neither','odds_ratio_HA','p_value','q_value_BH','direction']]


,stratum,gene1,gene2,both_mut,gene1_only,gene2_only,neither,odds_ratio_HA,p_value,q_value_BH,direction
224,site:eye,GNAQ,GNA11,11,601,526,0,0.000018,7.067224e-318,7.420585e-316,mutual exclusivity
0,pan-cancer,GNAS,GNAQ,30,2658,1241,2223,0.020218,1.002149e-309,1.052256e-307,mutual exclusivity
1,pan-cancer,GNAS,GNA11,30,2658,835,2629,0.035536,2.828052e-184,1.484727e-182,mutual exclusivity
14,pan-cancer,GNAQ,GNA11,32,1239,833,4048,0.125509,5.231651e-53,1.831078e-51,mutual exclusivity
2,pan-cancer,GNAS,GNA13,14,2674,261,3203,0.064251,3.004820e-49,7.887654e-48,mutual exclusivity


## Notes / limitations

- Sample unit is COSMIC `ID_sample` (not patient-level); COSMIC does not reliably expose patient identity.
- Absence of a mutation record for a gene in a sample is treated as wild-type; this cannot distinguish a true negative from a gene that was simply not sequenced/reported for that sample.
- Benjamini–Hochberg correction is applied separately within each stratum's 105 tests.
- Pan-cancer mutual-exclusivity signals involving GNAS/GNAQ/GNA11 are confounded by differing tissue-of-origin distributions across COSMIC and should be interpreted with the tissue-stratified results above, not the pan-cancer numbers alone.
- Full results (all 2,835 tests, all strata): `results/CoMutation_Statistical_Results_FULL.csv`.